In [16]:
from glob import glob
from cmcmultimodal.core.io import mat2nii
from pathlib import Path
import os

my_files = glob('/Users/Vasilis/CMC_data/Moe/PSOCT/Orientation/Slice*.mat')
highres_files = []
lowres_files = []
for f in my_files:
    nii_highres, nii_lowres = mat2nii(f, nii_lowres_file=os.path.join(Path(f).parent,'lowres/',
                                      Path(f).name.replace('.mat','.nii.gz')), downsample=10)
    highres_files.append(nii_highres)
    lowres_files.append(nii_lowres)


In [ ]:
# TODO delete this section after all data have been flipped and analysis worked correctly
# Only to be used once!!!
import subprocess
import glob
from pathlib import Path

inp_path = Path('/Users/Vasilis/CMC_data/Moe/PSOCT')
modalities = []#['Cross', 'Retardance', 'Orientation', 'Reflectivity']

for mod in modalities:
    files = (inp_path / mod).glob('Slice_*_En*.nii.gz')
    for f in files:
        input  = f
        output = f

        subprocess.run(["fslswapdim",
                        input,
                        "-x", "y", "z",
                        output])
    
    files = (inp_path / mod / 'lowres').glob('Slice_*_En*.nii.gz')
    for f in files:
        input  = f
        output = f

        subprocess.run(["fslswapdim",
                        input,
                        "-x", "y", "z",
                        output])


In [3]:
# Mask Retardance by Cross
import subprocess
import os
from pathlib import Path
from fsl.wrappers import fslmaths#, LOAD

inp_path = Path('/Users/Vasilis/CMC_data/Moe/PSOCT')
target_mod  = 'Retardance'
mask_mod    = 'Cross'
out_path    = inp_path / (target_mod[:3] + '_masked_by_' + mask_mod[:3]) / 'lowres'
os.makedirs(out_path, exist_ok=True)

files = (inp_path / target_mod / 'lowres').glob('Slice_*_En*.nii.gz')
for f in files:
    mask_file = inp_path / mask_mod / 'lowres' / f.name.replace('EnR','EnCr')

    # mask = fslmaths(mask_file).bin().run(LOAD)

    fslmaths(f).mas(mask_file).run(out_path / f.name)


In [ ]:
# Process all MOE slides

from cmcmultimodal.core.proc import psoct
from pathlib import Path

datadir = '/Users/Vasilis/CMC_data/Moe'
output_path = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_test'
mri_ref = '/Users/Vasilis/CMC_data/Moe/MRI/dtifit_FA.nii.gz'
seq_params = '/Users/Vasilis/CMC_data/Moe/PSOCT/seq_params.json'

my_data = psoct(Path(datadir), seq_params, psoct_reg_mod='Cross', mri_reg_mod='Ret_masked_by_Cro', verbose=True)

my_data.run_pipeline(other_images=['Cross', 'Retardance', 'Orientation'], 
                                    output_path=output_path, 
                                    mri_ref=mri_ref, 
                                    downsample=1, 
                                    bad_slides=[],
                                    fnirt=True)

In [ ]:
from pathlib import Path
from cmcmultimodal.utils.validate_results import compare_results_folder

ref_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_cc_Ret')
est_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_cc_Ret_new')

compare_results_folder(ref_path, est_path, subdir=True)

In [2]:
from pathlib import Path
import subprocess
import os

inp_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_nl/')
os.makedirs(inp_path / 'Ret_split', exist_ok=True)

subprocess.run(['fslsplit',
                inp_path / 'Ret_masked_by_Cro_slide_deck.nii.gz',
                inp_path / 'Ret_split/vol',
                '-y'])

CompletedProcess(args=['fslsplit', PosixPath('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_nl/Ret_masked_by_Cro_slide_deck.nii.gz'), PosixPath('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_nl/Ret_split/vol'), '-y'], returncode=0)

In [ ]:
# Code to resample the PSOCT slices and turn them into a 3d slide deck

from fsl.wrappers.avwutils  import fslmerge
import glob, os
import numpy as np
from pathlib import Path
from fsl.wrappers                import flirt
from fsl.data.image              import Image
import dask.multiprocessing
import multiprocessing
import shutil

modality = 'Reflectivity'

inp_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_nl/')

out_file = inp_path / (modality[0:3] + '_slide_deck')
inp_files = sorted(glob.glob(str(inp_path / (modality + '/lowres/Slice*hdr.nii.gz'))))
ref_files = sorted(glob.glob(str(inp_path / 'Ret_split/*.nii.gz')), reverse=True)

slide_numbers = [int(Path(f).name.split('_')[1]) for f in inp_files]

missing_slides = list(set(np.arange(min(slide_numbers), max(slide_numbers)+1)) - set(slide_numbers))
missing_slides = list(map(int, missing_slides))

slide_arr = np.array(slide_numbers)
raw_files = inp_files.copy()
for m in sorted(missing_slides):
    # slide_numbers = [int(Path(f).name.split('_')[1]) for f in inp_files]

    # nearest slide before
    before = slide_arr[(slide_arr - m) < 0]
    if before.size == 0:
        before = np.inf
    else:
        before = before[np.argmin(np.abs(before-m))]
    # nearest slide after
    after = slide_arr[(slide_arr - m) > 0]
    if after.size == 0:
        after = np.inf
    else:
        after = after[np.argmin(np.abs(after-m))]
    # If both are Inf (logically impossible but could happen if slide_numbers is empty), raise an error
    if np.isinf(before) and np.isinf(after):
        raise ValueError(f"No available slide before or after missing slide {m}")
    if np.abs(m-before) < np.abs(after-m):
        inp_files.insert(m-1, raw_files[np.where(slide_arr == before)[0][0]])
        ref_files[m-1] = ref_files[m-2]
    else:
        inp_files.insert(m-1, raw_files[np.where(slide_arr == after)[0][0]])#np.where(slide_arr == after)[0][0]])
        
for m in sorted(missing_slides, reverse=True):
    # nearest slide before
    before = slide_arr[(slide_arr - m) < 0]
    if before.size == 0:
        before = np.inf
    else:
        before = before[np.argmin(np.abs(before-m))]
    # nearest slide after
    after = slide_arr[(slide_arr - m) > 0]
    if after.size == 0:
        after = np.inf
    else:
        after = after[np.argmin(np.abs(after-m))]
    # If both are Inf (logically impossible but could happen if slide_numbers is empty), raise an error
    if np.isinf(before) and np.isinf(after):
        raise ValueError(f"No available slide before or after missing slide {m}")
    if np.abs(m-before) >= np.abs(after-m):
        ref_files[m-1] = ref_files[m]


os.makedirs(out_file.parent / (modality[0:3] +'_temp_files'), exist_ok=True)
jobs = []
for i in range(len(inp_files)):
    inp = inp_files[i]
    ref = ref_files[i]
    filename = 'vol_' + str(i).zfill(3) + '.nii.gz'
    out_filename = out_file.parent / (modality[0:3] + '_temp_files') / filename
    jobs.append(dask.delayed(flirt)(inp, ref, out=out_filename, usesqform=True, applyxfm=True, twod=True))
dask.compute(jobs)


out_files = sorted(glob.glob(str(out_file.parent / (modality[0:3] + '_temp_files/*.nii.gz'))), reverse=True)
fslmerge('y', out_file, *out_files)
shutil.rmtree(out_file.parent / (modality[0:3] + '_temp_files'))


In [ ]:
# Code to register the 3d slide decks to MRI space (non-linear case)

from fsl.wrappers.fnirt import applywarp

modality = 'Ref'

inp_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_nl/')
warp = inp_path / 'PSOCT_to_MRI_warpfield.nii.gz'
mri_ref = inp_path / 'dtifit_FA.nii.gz'

inp_file = inp_path / (modality + '_slide_deck')
out_file = inp_path / (modality + '_slide_deck_in_MRI')


applywarp(inp_file, mri_ref, out_file, warp=warp)

{}

In [ ]:
import subprocess
inp_folder = out_file.parent / (modality[0:3] +'_temp_files')
files = sorted(inp_folder.glob('*.nii.gz'))
subprocess.run(['fslmaths',
                files[0],
                '-mul', '0',
                str(out_file)])
cmd = ['fslmaths', str(out_file)]
for i in files:
    cmd.extend(['-add', str(i)])
cmd.append(str(out_file))
subprocess.run(cmd)

In [ ]:
from fsl.utils.image.resample import resampleToReference
from fsl.data.image              import Image
from pathlib import Path

inp_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross_nl/')

src_filename = Image(inp_path / 'Retardance/lowres/Slice_060_EnR_downsample_10_hdr.nii.gz')
tgt_filename = Image(inp_path / 'Ret_slide_deck.nii.gz')
out_filename = inp_path / 'resampled_slices/slide_060_new.nii.gz'
results = resampleToReference(src_filename, tgt_filename,)
Image(results[0], header = tgt_filename.header).save(out_filename)

In [29]:
from fsl.wrappers           import flirt

mri_ref = '/Users/Vasilis/CMC_data/Moe/MRI/dtifit_FA.nii.gz'
outfile = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/dtifit_FA_in_Ret_PSOCT'
matfile = '/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross/MRI_to_Ret_PSOCT.mat'

flirt(src=mri_ref, ref=out_file, out=outfile, omat=matfile, dof=12, interp='spline')


{}

In [3]:
import json
import numpy as np
from pathlib import Path

output_path = Path('/Users/Vasilis/Downloads/CMC_results/Moe_flirt_Cross')

with open(output_path / "abs_mat.json") as f:
    data = json.load(f)

abs_mat = {int(k): np.array(v) for k, v in data.items()}